# Bloco 1: Instalação e Configuração (API Keys e Variáveis Globais)
Neste bloco são importadas as bibliotecas, credenciais de acesso às APIS de inteligência artificial (Google Gemini) e definida a estrutura básica com caminhos do projeto.

In [ ]:
# Descomente a linha abaixo para executar instalações iniciais necessárias
# !pip install pandas openpyxl scikit-learn google-generativeai tqdm numpy

import os
import json
import time
import pandas as pd
import numpy as np
from tqdm import tqdm
import google.generativeai as genai
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --- VARIÁVEIS GLOBAIS DE MODELO ---
# Altere esta variável de configuração para realizar o toggle para GPT posteriormente.
MODEL_NAME = "gemini-2.5-flash" 

# --- CONFIGURAÇÃO DE CREDENCIAIS GEMINI ---
GOOGLE_API_KEY = ""
genai.configure(api_key=GOOGLE_API_KEY)

# --- CAMINHOS DINÂMICOS DE ARQUIVOS ---
BASE_PATH = r"C:\Users\stgab\codigos_git\Grupo_2_atividades\projetos Antigravity"
INPUT_FILE = os.path.join(BASE_PATH, "BASE_DE_EXEMPLO.xlsx")
CHECKPOINT_FILE = os.path.join(BASE_PATH, "processamento_checkpoint.csv")
TAXONOMIA_FILE = os.path.join(BASE_PATH, "taxonomia_macros.json")
OUTPUT_FILE = os.path.join(BASE_PATH, "BASE_CLASSIFICADA.xlsx")

# --- CONFORTO E PARAMETRIZAÇÕES ---
SIMILARITY_THRESHOLD = 0.90 # 90% de conformidade textual pulará o LLM
CHECKPOINT_INTERVAL = 10 # Intervalo de gravações por lote


# Bloco 2: Funções de Similaridade (scikit-learn) e Gestão de Taxonomia (JSON)
Lógica responsável por gerar reaproveitamento local das descrições idênticas sem onerar tokens, e realizar a ponte entre o LLM de taxonomia (Macros) salvos num arquivo para padronização da rede.

In [ ]:
def carregar_taxonomia():
    '''Carrega arquivo fixo contendo os macros/micros do JSON como fonte de diretriz.'''
    if os.path.exists(TAXONOMIA_FILE):
        with open(TAXONOMIA_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {}

def salvar_taxonomia(taxonomia_atualizada):
    '''Registra novas subcategorias validadas pela API garantindo a integridade dos dados.'''
    with open(TAXONOMIA_FILE, 'w', encoding='utf-8') as f:
        json.dump(taxonomia_atualizada, f, ensure_ascii=False, indent=4)

def buscar_similaridade(nova_descricao, text_historico, class_historico):
    '''
    Avalia em tempo de voo com TF-IDF se essa transação repete padrão igual a anterior.
    Se o grau de similaridade (Cosine Similarity) ultrapassar os 90%, copia-se a classificação.
    '''
    if not text_historico:
        return None
        
    textos = text_historico + [nova_descricao]
    vectorizer = TfidfVectorizer()
    
    try:
        tfidf_matriz = vectorizer.fit_transform(textos)
        # Relacionamento de vetores calculando correlação da penultima em relação ao todo.
        similaridades = cosine_similarity(tfidf_matriz[-1], tfidf_matriz[:-1]).flatten()
        
        index_maximo = similaridades.argmax()
        grau_confianca = similaridades[index_maximo]
        
        if grau_confianca >= SIMILARITY_THRESHOLD:
            return class_historico[index_maximo]
            
    except Exception as erro:
        print(f"Alarme no algoritmo lógico do Cosine: {erro}")
        
    return None


# Bloco 3: Wrapper do LLM (Preparado para Azure/GPT/Gemini)
Encapsula o prompt JSON, enviando instruções rígidas pelo System. Todo o tratamento de extração do Response é embutido na base.

In [ ]:
def consultar_llm(prompt_diretivo):
    '''
    Este é a Wrapper Global. 
    => Para trocar de API Google para OpenAI (ChatGPT ou Azure GPT), basta fazer a 
       chamada 'openai.ChatCompletion.create' dentro do bloco 'elif "gpt"'.
    '''
    if "gemini" in MODEL_NAME.lower():
        # Setup do Gemini
        modelo_agente = genai.GenerativeModel(MODEL_NAME)
        resposta_retornada = modelo_agente.generate_content(prompt_diretivo)
        return resposta_retornada.text
    elif "gpt" in MODEL_NAME.lower():
        # Lógica pronta de espelhamento estrutural OpenAI.
        # response = openai.ChatCompletion.create(model=MODEL_NAME, messages=[...])
        raise NotImplementedError("GPT ainda não foi ativado neste provedor global de abstração. Importe OpenAI.")
    else:
        raise RuntimeError("Definição inválida do LLM_MODEL.")

def processar_texto_com_llm(apontamento_banco, desc_banco, arvore_taxonomia):
    '''
    Processa com obrigatoriedade estrita em JSON a entrega do Macro / Micro avaliada pela IA.
    '''
    contexto_tax_json = json.dumps(arvore_taxonomia, ensure_ascii=False)
    
    # SYSTEM PROMPT ROBUSTO: Regras duras de negócio para devolver exatamente em string JSON.
    prompt = f'''
Você atuará como um categorizador bancário corporativo.
Ao analisar a situação, retorne somente duas variáveis: Diga-me qual é a categoria maior (macro) e a específica (micro).

>>> INFORMAÇÕES DO APONTAMENTO ATUAL:
[APONTAMENTO]: {apontamento_banco}
[DESCRICAO]: {desc_banco}

>>> HISTÓRICO LOCAL DOS MACROS (Utilize estes primeiro antes de inventar):
{contexto_tax_json}

>>> SUA REGRA CRÍTICA:
Forneça ESTRITAMENTE E UNICAMENTE o arquético visual do JSON abaixo. Absolutamente NENHUMA outra palavra ou crase MD é premitida, pois vou dar um JSON Loads direto:
{{
   "macro": "nome_do_macro",
   "micro": "nome_do_micro"
}}
'''
    
    # Dispara e aguarda API Generativa
    bloco_conteudo = consultar_llm(prompt)
    
    # Tratamento contra Markdown backticks do Google ```json
    texto_raw_limpo = bloco_conteudo.strip()
    if texto_raw_limpo.startswith("```json"):
        texto_raw_limpo = texto_raw_limpo[7:]
    if texto_raw_limpo.startswith("```"):
        texto_raw_limpo = texto_raw_limpo[3:]
    if texto_raw_limpo.endswith("```"):
        texto_raw_limpo = texto_raw_limpo[:-3]
        
    try:
        dados_classificados = json.loads(texto_raw_limpo.strip())
        return {
            "macro": dados_classificados.get("macro", "Desconhecido"), 
            "micro": dados_classificados.get("micro", "Desconhecido")
        }
    except json.JSONDecodeError:
        print(f"\nIncoerência de Parser detectada no Parser: {bloco_conteudo}")
        return {"macro": "Erro_LLM", "micro": "Erro_LLM"} 


# Bloco 4: Lógica de Checkpoint e Retomada de Processamento
Esta robustez possibilita retomar do checkpoint exato de onde caiu se faltar energia ou exceder APIs (Exponential Backoff).

In [ ]:
def invocar_com_tolerancia_rate_limit(*args, limite_tentativas=3):
    '''
    Estrutura de resiliência. Atua por Exponential Backoff se a API gerar exceções 429 ou 503.
    Dobra o sleep time conforme caem requests.
    '''
    segundos_espera = 2
    for ronda in range(limite_tentativas):
        try:
            return processar_texto_com_llm(*args)
        except Exception as error_bruto:
            print(f"\n[*] Erro da API: Fallback ativado ({ronda+1}/{limite_tentativas}) | Aguardando {segundos_espera}s -> {error_bruto}")
            time.sleep(segundos_espera)
            segundos_espera *= 2 # Backoff dinâmico
            
    return {"macro": "Lim_API", "micro": "Lim_API"}

def boot_checkpoint_ou_criar_novo():
    '''Monitora a presença de arquivos de parada (checkpoints) para a retomada.'''
    df_base = pd.read_excel(INPUT_FILE)
    
    if os.path.exists(CHECKPOINT_FILE):
        print(f"Continuando leitura incremental do último salvamento {CHECKPOINT_FILE}...")
        df_recup = pd.read_csv(CHECKPOINT_FILE)
        
        # Filtros das coisas que conseguimos concluir.
        # Estas descrições retroalimentam instantaneamente o TF-IDF.
        ja_processado = df_recup[df_recup['TAG_CONTROLE'] == 'Feito']
        txt_historico = ja_processado['DESCRICAO'].astype(str).tolist()
        class_historico = [{'macro': mac, 'micro': mic} for mac, mic in zip(ja_processado['macro'], ja_processado['micro'])]
        
        return df_recup, txt_historico, class_historico
        
    else:
        print("Trabalho primário. Iniciando carga completa desde o Teto.")
        df_base['macro'] = None
        df_base['micro'] = None
        df_base['TAG_CONTROLE'] = 'Aberto...'
        return df_base, [], []


# Bloco 5: Loop Principal de Processamento Incremental (5.000 registros seguros)
Processa cada campo em batches. Salava incrementais. Utiliza Barra de carregamento nativa gráfica paralela.

In [ ]:
dataframe_global, pilha_descricoes, pilha_classificacoes = boot_checkpoint_ou_criar_novo()
base_taxonomia = carregar_taxonomia()

# Encontra alvos do Lote
filas_abertas = dataframe_global[dataframe_global['TAG_CONTROLE'] == 'Aberto...'].index
print(f"Faltam analisar {len(filas_abertas)} Linhas bancárias.")

batch_contador = 0

for idx in tqdm(filas_abertas, desc="Processamento pela Rede Neural"):
    row_local = dataframe_global.loc[idx]
    v_apontamento = str(row_local.get('APONTAMENTO', ''))
    v_descricao = str(row_local.get('DESCRICAO', ''))
    
    # ETAPA 1: OTIMIZAÇÃO (Se tivermos texto parecido de apontamentos repetitivos. Ignora o LLM.)
    resultado = buscar_similaridade(v_descricao, pilha_descricoes, pilha_classificacoes)
    
    # ETAPA 2: Acionamento da IA Inteligente (LLM) apenas em contextos descolados.
    if not resultado:
        resultado = invocar_com_tolerancia_rate_limit(v_apontamento, v_descricao, base_taxonomia)
        
        mac_name, mic_name = resultado.get('macro'), resultado.get('micro')
        
        # ETAPA 3: Persistência local do vocabulário novo para contextualizar da base (RAG Base)
        if "Erro" not in mac_name and "Lim_API" not in mac_name:
            if mac_name not in base_taxonomia:
                base_taxonomia[mac_name] = []
                
            if mic_name not in base_taxonomia[mac_name]:
                base_taxonomia[mac_name].append(mic_name)
                salvar_taxonomia(base_taxonomia)
                
        # Respiro da API (Throttle leve para prevenção de Block Limits)
        time.sleep(1)
        
    # Alimentação Imediata ao pool TF-IDF Ponderal
    pilha_descricoes.append(v_descricao)
    pilha_classificacoes.append(resultado)
    
    # Edição Direta no Frame na sua fila
    dataframe_global.at[idx, 'macro'] = resultado.get('macro')
    dataframe_global.at[idx, 'micro'] = resultado.get('micro')
    dataframe_global.at[idx, 'TAG_CONTROLE'] = 'Feito'
    
    # ETAPA 4: Acionamento de Gatilho de Backup Local via Threshold
    batch_contador += 1
    if batch_contador >= CHECKPOINT_INTERVAL:
        dataframe_global.to_csv(CHECKPOINT_FILE, index=False)
        batch_contador = 0

# Garantia da Gravação Final caso sobre restos antes dos 10 de lotes...
dataframe_global.to_csv(CHECKPOINT_FILE, index=False)
print("Pipeline Completa!")


# Bloco 6: Exportação & Compilação
Ao concluir tudo do checkpoint, destrói e recria o painel em Excel puro para a operação real.

In [ ]:
# Validação se existe algo inacabado do processamento do laço
elementos_em_falta = dataframe_global[dataframe_global['TAG_CONTROLE'] == 'Aberto...']

if len(elementos_em_falta) == 0:
    # Remoção das flags de desenvolvedor 
    base_limpa_final = dataframe_global.drop(columns=['TAG_CONTROLE'])
    
    # Extensão Real Excel
    base_limpa_final.to_excel(OUTPUT_FILE, index=False)
    print(f"Trabalho perfeito! XLSX unificado exportado localizado em: {OUTPUT_FILE}")
else:
    print(f"AVISO: O Processo parou ainda e faltam {len(elementos_em_falta)} registros. Repita a operação!")
